<p align="center">
    <span style="font-size:2.5em; font-weight:bold;">
        eFleetPlan - Optimal infrastructure and fleet operation of electric LCV
    </span>
</p>

<p align="center">
    <span style="font-size:1.5em; font-weight:bold;">
        Carolina Gil Ribeiro, Jagruti Thakur
    </span>
</p>

## 1. Fleet Operations Simulation

### 1.1. Importing dependencies

In [ ]:
import sys, os, time
from pathlib import Path

# ── Project root detection ─────────────────────────────────────────────────
def _find_project_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "src").exists():      return parent
        if (parent / "pyproject.toml").exists(): return parent
        if (parent / "setup.py").exists(): return parent
    raise FileNotFoundError(
        f"Could not find project root from '{start}'. "
        "Expected 'src/', 'pyproject.toml', or 'setup.py' at the root."
    )

project_root = _find_project_root(Path(os.getcwd()))
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root))

# ── Imports ────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pydantic import ValidationError

from config._0_supportfiles.config_loader_schedule import (
    load_config, VehicleConfig, RunConfig,
)
from src.efleetplan._1_fleetoperation_simulation.schedule_generation import (
    ScheduleGenerator, generate_fleet_schedules,
)
from src.efleetplan._1_fleetoperation_simulation.generate_graphs import generate_graphs

print("All imports OK")


### 1.2. Configurations

#### 1.2.1 Environment parameters

Edit `env.yaml` and `run_FleetSchedule_Config.yaml` to change simulation parameters.

In [ ]:
# ── Load environment + fleet configuration ─────────────────────────────────
# Edit env.yaml / run_FleetSchedule_Config.yaml to change parameters.

config_dir = os.path.join(project_root, "config")

env, run, predefined = load_config(
    env_yaml      = os.path.join(config_dir, "Simple_test", "env_TEST.yaml"),
    run_yaml      = os.path.join(config_dir, "Simple_test", "run_FleetSchedule_Config_TEST.yaml"),
    predefined_dir= os.path.join(config_dir, "predefined"),
)

print("Environment and run configuration loaded successfully:")
print(f"  Seed:       {env.seed}")
print(f"  Period:     {env.gen_start_date} → {env.gen_end_date}")
print(f"  Frequency:  {env.freq}")
print(f"  Run:        {run.schedule_name}  ({run.n_vehicles} vehicles, company={run.company_type})")


#### 1.2.2 Fleet parameters

Schedule types: **Type A** (continuous) and **Type B** (two-part with break).

In [ ]:
# ── Schedule parameters ────────────────────────────────────────────────────
for sched_name, count in run.schedule_mix.items():
    if count == 0:
        continue
    sc = predefined.get_schedule(sched_name, run.custom_schedule)
    print(f"--- ScheduleConfig: '{sched_name}' ({count} vehicles) ---")
    for field, value in sc.model_dump(exclude_none=True).items():
        print(f"  {field}: {value}")
    print()

# ── Vehicle + company parameters ───────────────────────────────────────────
for veh_name, count in run.vehicle_mix.items():
    if count == 0:
        continue
    vc = predefined.get_vehicle(veh_name, run.custom_vehicle)
    print(f"--- VehicleConfig: '{veh_name}' ({count} vehicles) ---")
    for field, value in vc.model_dump().items():
        print(f"  {field}: {value}")
    print()

cc = predefined.get_company(run.company_type, run.custom_company)
print(f"--- CompanyConfig: '{run.company_type}' ---")
for field, value in cc.model_dump().items():
    print(f"  {field}: {value}")


#####   - Company type
The values related to company type are defined based on the findings of a survey on light goods vehicles in Sweden conducted by Transport Analysis in 2022. 

![image.png](attachment:image.png)

notes: Type of transport is defined as the primary use of the LCV and are described as follows: distribution transport for goods or commodities transports with several stops for loading and unloading along the way; Line haul goods or commodities transports, directly from one place to another;  Crafts and services with goods transports include trips for craft or service vehicle that include goods or merchandise to be used or installed in the work; and crafts and services without goods correspond to craft or service trips without goods or merchandise.

![image-2.png](attachment:image-2.png)

notes: Commodity groups were formulated in Light Goods Vehicles 2022 survey to capture the type of goods commonly transported identified by the vehicles.

### 1.3. Config validation tests

In [ ]:
# ── Config validation tests ────────────────────────────────────────────────

# Test 1: negative battery capacity
try:
    VehicleConfig(
        consumption_mean=-0.2, consumption_std=0.1,
        consumption_min=0.1,   consumption_max=0.4,
        total_cons_clip=45,    battery_capacity=45, charging_power=80,
    )
    print("FAIL – negative value not rejected")
except ValidationError as e:
    print(f"Test 1 PASSED: {e.errors()[0]['msg']}")

# Test 2: typo in field name
try:
    VehicleConfig(
        consumptoin_mean=0.2, consumption_std=0.1,
        consumption_min=0.1,  consumption_max=0.4,
        total_cons_clip=45,   battery_capacity=45, charging_power=80,
    )
    print("FAIL – typo not rejected")
except ValidationError as e:
    print(f"Test 2 PASSED: {e.errors()[0]['msg']}")

# Test 3: mismatched fleet mix
try:
    RunConfig(
        schedule_name="test", n_vehicles=50,
        schedule_mix={"typea": 30},
        vehicle_mix={"renault": 50},
        company_type="distribution",
    )
    print("FAIL – mismatched mix not rejected")
except ValidationError as e:
    print(f"Test 3 PASSED: {e.errors()[0]['msg']}")


### 1.4. Generation of fleet operational schedules

In [ ]:
# ── Generate fleet schedules ───────────────────────────────────────────────
# Add more run YAML filenames to run_files to generate multiple fleets.

run_files = ["run_FleetSchedule_Config_TEST.yaml"]
all_results = {}

for run_file in run_files:
    env, run, predefined = load_config(
        env_yaml      = os.path.join(config_dir, "Simple_test", "env_TEST.yaml"),
        run_yaml      = os.path.join(config_dir, "Simple_test", run_file),
        predefined_dir= os.path.join(config_dir, "predefined"),
    )
    
    print("consumption_factor_file:", env.consumption_factor_file)
    print("exists:", env.consumption_factor_file.exists())

    print(f"--- Generating: {run.schedule_name} ({run_file}) ---")
    print(f"    {run.n_vehicles} vehicles, company={run.company_type}")

    t0 = time.time()
    schedule = generate_fleet_schedules(env, run, predefined)
    print(f"    {len(schedule):,} rows in {time.time()-t0:.1f}s\n")

    all_results[run.schedule_name] = schedule


### 1.5. Results validation

In [ ]:
# ── Results validation ─────────────────────────────────────────────────────
for name, schedule in all_results.items():
    print(f"{'='*60}\nValidation: {name}\n{'='*60}")

    expected_steps = len(pd.date_range(
        start=env.gen_start_date, end=env.gen_end_date, freq=env.freq
    ))
    steps = schedule.groupby("VehicleID").size()
    assert (steps == expected_steps).all(), "Timestep count mismatch!"
    print(f"✓ All {steps.nunique()} vehicles have {expected_steps} timesteps")

    assert (schedule["Distance_km"]   >= 0).all(), "Negative distances!"
    print("✓ No negative distances")
    assert (schedule["Consumption_kWh"] >= 0).all(), "Negative consumption!"
    print("✓ No negative consumption")
    assert schedule["Location"].isin([0, 1]).all(), "Invalid location values!"
    print("✓ Location values valid (0=driving, 1=depot)")

    for col, label in [("ScheduleType", "Schedule"), ("VehicleType", "Vehicle")]:
        counts = schedule.groupby(col)["VehicleID"].nunique()
        print(f"  {label} types: " + ", ".join(f"{k}: {v}" for k, v in counts.items()))

    schedule["date"] = pd.to_datetime(schedule["date"])
    daily = schedule.groupby([schedule["date"].dt.date, "VehicleID"])["Distance_km"].sum()
    print(f"  Daily distance: mean={daily.mean():.1f} km  std={daily.std():.1f} km  "
          f"min={daily.min():.1f} km  max={daily.max():.1f} km\n")

print(f"All {len(all_results)} schedule(s) passed ✓")


### 1.6. Post-processing and visualisation

In [ ]:
# ── Post-processing and visualisation ─────────────────────────────────────
for name in all_results:
    schedule_csv  = os.path.join(project_root, "data", "Output", name, f"{name}.csv")
    output_folder = os.path.join(project_root, "data", "Output", name)
    generate_graphs(schedule_csv=schedule_csv, folder=name, output_folder=output_folder)
